In [1]:
import os
import warnings 
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import anndata as ad
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from anndata import AnnData
from natsort import natsorted
from tqdm.notebook import tqdm
from scipy import stats
from adjustText import adjust_text
import matplotlib.patches as mpatches

In [2]:
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind
from scipy.stats import mannwhitneyu
import itertools

In [3]:
plt.rcParams['pdf.fonttype'] = 42

In [4]:
base_path = '/stanley/WangLab/Data/Analyzed/2024-12-02-Mingrui-SCZ/expr'

input_path = os.path.join(base_path, 'cell type classification')

out_path = os.path.join(base_path, 'DE analysis')
if not os.path.exists(out_path):
    os.mkdir(out_path)
    
fig_path = os.path.join('/stanley/WangLab/Data/Analyzed/2024-12-02-Mingrui-SCZ/figures','manuscript figure','SI')
if not os.path.exists(fig_path):
    os.mkdir(fig_path)

# sc.settings.figdir = fig_path

In [5]:
sc._settings.settings._vector_friendly=True

In [6]:
# adata = sc.read_h5ad(os.path.join(input_path, '2025-01-09-all-sample-cell-typing-lv3-subcluster.h5ad'))
# adata = sc.read_h5ad(os.path.join(input_path, '2025-02-27-all-sample-cell-typing-lv3-subcluster-rgn-label.h5ad'))
# adata = sc.read_h5ad(os.path.join(input_path, '2025-04-08-finalized-celltyping.h5ad'))
adata = sc.read_h5ad(os.path.join(base_path, '2025-04-11-finalized-ct-rgn.h5ad'))

# Fig.1

## Fig.1c Cell type classification

In [29]:
sc._settings.settings._vector_friendly=True

In [30]:
sc.set_figure_params(dpi_save=300)
sc.settings.figdir = fig_path


In [23]:
sc.pl.umap(adata, color='level_2', legend_loc=None, frameon=False, 
           title='', save='_level_2_no_legend_large.pdf')


In [9]:

# Plot UMAP with cluster labels w/ new color
sc.pl.umap(adata, color='level_3', legend_loc='right margin',
           legend_fontsize=12, legend_fontoutline=2, frameon=False, 
           title='', save='_level_3.pdf')


In [17]:
adata.obs['level_3'].cat.categories

In [16]:
l1_colors = [adata.uns['level_1_color_dict'][i] for i in adata.obs['level_1'].cat.categories]
l2_colors = [adata.uns['level_2_color_dict'][i] for i in adata.obs['level_2'].cat.categories]
l3_colors = [adata.uns['level_3_color_dict'][i] for i in adata.obs['level_3'].cat.categories]

level_1_pl = sns.color_palette(l1_colors)
level_2_pl = sns.color_palette(l2_colors)
level_3_pl = sns.color_palette(l3_colors)

In [ ]:

sc.pl.umap(adata, color='level_3', legend_loc=None, frameon=False, 
           title='', save='_level_3_no_legend.pdf')

sc.pl.umap(adata, color='level_3', legend_loc=None, frameon=False, 
           title='', save='_level_3_no_legend.png')


In [14]:
# Create a new column level_3_abrr with prefix before '-'
adata.obs['level_3_abbr'] = adata.obs['level_3'].apply(lambda x: x.split('-')[0])


## Fig.1d Region label

In [15]:
sc._settings.settings._vector_friendly=True
sc.set_figure_params(dpi_save=300)
sc.settings.figdir = fig_path

In [ ]:
sc.pl.spatial(adata, color='region_label', legend_loc='right margin',
           legend_fontsize=12, legend_fontoutline=2, frameon=False, 
           title='', save='_rgn.pdf')

In [ ]:
sc.pl.spatial(adata, color='region_label', legend_loc='right margin',
           legend_fontsize=12, legend_fontoutline=2, frameon=False, 
           title='', save='_rgn.pdf')

In [91]:
sample_list = ['sample2','sample6','sample15']

In [132]:
adata.obs['global_x'].max()

In [133]:
for i, sample in enumerate(sample_list):
    adata_sample = adata[adata.obs['sample'] == sample].copy()
    print(sample, adata_sample.obs['global_x'].max())

In [ ]:
num_rows = 1
num_cols = 3
fig, axes = plt.subplots(num_rows, num_cols, figsize =(num_cols * 8, num_rows*6))

axes_flat = axes.flatten()

for i, sample in enumerate(sample_list):
    adata_sample = adata[adata.obs['sample'] == sample].copy()
    ax = axes_flat[i]  # Explicitly reference the axis for this iteration

    sc.pl.spatial(adata_sample, color='level_3', spot_size=150, title=f'coronal position {i+1}', show=False, ax=ax)

    if i in [0, 1]:  
        ax.invert_xaxis()
        ax.invert_yaxis()

In [ ]:
adata_sample = adata[(adata.obs['sample'] == 'sample2')&(adata.obs['level_1'] != 'Mix')].copy()

sc.pl.spatial(adata_sample, color='level_3', s,legend_loc='right margin',
        legend_fontsize=12, legend_fontoutline=2, frameon=False, 
        title='', )


In [12]:
level_3_order = adata.obs['level_3'].unique().tolist()
level_3_order = ['Mix'] + [x for x in level_3_order if x != 'Mix']

In [ ]:

for i, sample in enumerate(sample_list):
    adata_sample = adata[adata.obs['sample'] == sample].copy()

    sc.pl.spatial(adata_sample, color='level_3', spot_size=150,legend_loc='right margin',
           legend_fontsize=12, legend_fontoutline=2, frameon=False, 
           title='', save=f'{sample}_ct_spatial.pdf',groups=level_3_order)


In [ ]:
for i, sample in enumerate(sample_list):
    adata_sample = adata[adata.obs['sample'] == sample].copy()

    # Split Mix and non-Mix cells
    adata_mix = adata_sample[adata_sample.obs['level_3'] == 'Mix'].copy()
    adata_other = adata_sample[adata_sample.obs['level_3'] != 'Mix'].copy()

    # Plot Mix cells with low alpha (grey)
    ax = sc.pl.spatial(
        adata_mix,
        color='level_3',
        palette={'Mix': '#d3d3d3'},  # light grey
        spot_size=300,
        alpha_img=0.3,
        frameon=False,
        title='',
        legend_loc=None,
        show=False
    )
    ax = ax[0]
    # Plot other cells on top with default coloring
    sc.pl.spatial(
        adata_other,
        color='level_3',
        spot_size=300,
        frameon=False,
        legend_loc='right margin',
        legend_fontsize=12,
        legend_fontoutline=2,
        title='',
        ax=ax,
        save=f'{sample}_ct_spatial_sz300.pdf',
        show=True
    )


In [67]:
adata.uns['level_3_color_dict']

In [ ]:
rgn_order = ['CTX_L2/3',
             'CTX_L4',
             'CTX_L5',
             'CTX_L6',
             'CTX_ILA2/3',
             'CTX_mPFC5',
             'CTXsp',
             'CTX_AI2/3',
             'CTX_PIR',
             'CTX_CA1',
             'CTX_CA3',
             'CTX_DG',
             'CTX_HIP',
             'STR',
             'PAL',
             'TH_1',
             'TH_2',
             'TH_EPI',
             'TH_RT',
             'HY',
             'FT',
             'VS',
             'MNG'
             ]


adata.obs['region_label'] = adata.obs['region_label'].cat.reorder_categories(rgn_order)

adata.uns['region_label_order'] = rgn_order


In [ ]:
region_label_color_dict = { 
    'CTX_L2/3': '#AEFF32',
    'CTX_L4': '#57E354',
    'CTX_L5': '#2AB529',
    'CTX_L6': '#486E21',
    'CTX_ILA2/3': '#FFEEB2',
    'CTX_mPFC5': '#F6C34C',
    'CTXsp':'#80FBD2',
    'CTX_AI2/3':'#a8e1eb',
    'CTX_PIR': '#cccccc',
    'CTX_CA1': '#8B09BE',
    'CTX_CA3': '#CB1587',
    'CTX_DG': '#A07CB3',
    'CTX_HIP': '#F7BDC5',
    'STR': '#00C1FD',
    'PAL': '#1f76b3',
    'TH_1': '#f78a88',
    'TH_2': '#EB9FA9',
    'TH_EPI': '#FFC6E5',
    'TH_RT': '#C85C0C',
    'HY': '#ed5e5b',
    'FT': '#101190',
    'VS': '#789AB3',
    'MNG': '#B1D3C3'
}
adata.uns['region_label_color_dict'] = region_label_color_dict

In [95]:
num_rows = 1
num_cols = 3
fig, axes = plt.subplots(num_rows, num_cols, figsize =(num_cols * 8, num_rows*6))

axes_flat = axes.flatten()

for i, sample in enumerate(sample_list):
    adata_sample = adata[adata.obs['sample'] == sample].copy()
    ax = axes_flat[i]  # Explicitly reference the axis for this iteration

    sc.pl.spatial(adata_sample, color='region_label', spot_size=300, palette=region_label_color_dict, title=f'coronal position {i+1}', show=False, ax=ax)

    if i in [0, 1]:  
        ax.invert_xaxis()
        ax.invert_yaxis()

plt.show()   

# SI2

In [ ]:
level_3_marker_dict ={}
for i in adata.obs['level_3'].cat.categories:
    if i == 'Mix' :
        continue
    if i == 'TEGLU_Mix':
        level_3_marker_dict[i] = ['Nrgn','Gria3']
    else:
        level_3_marker_dict[i] = [i.split('-')[-1].split('_')[0][1:],
                                i.split('-')[-1].split('_')[1][:-1]
                                ]


In [21]:
level_3_marker_dict

In [25]:
level_3_marker_list = []
for key in level_3_marker_dict.keys():
    level_3_marker_list= level_3_marker_list + level_3_marker_dict[key]

In [30]:
# remove duplicate in level_3_marker_list qand amin tain order
level_3_marker_list = list(dict.fromkeys(level_3_marker_list))

In [61]:
len(adata.obs['level_2'].cat.categories)

In [79]:
level_2_marker_dict ={
    'TEGLU': ['Nrgn','Slc17a7','Neurod6','Atp1a1'],
    'DGGRC': ['Gabra5','Prox1','Nr3c2','Neurod2'],
    'TEINH': ['Gad1','Gad2','Slc6a1','Pvalb'],
    'MSN': ['Drd1','Drd2','Ppp1r1b','Pde10a'],
    'DEGLU': ['Prkcd','Tcf7l2','Plekhg1','Synpo2'],
    'PEP': ['Dlk1','Ly6h','Peg3','Scg2'],
    'HABCHO': ['Gabbr1','Lrrc55','Lhx9','Nrp2'],
    'AC': ['Gja1', 'Fam107a', 'Clu', 'Aqp4'],
    'OLG': ['Mbp','Mal','Aplp1','Car2'],
    'OPC': ['Cspg5','Pdgfra','Cacng4','Ptprz1'],
    'MGL': ['Csf1r','Ctss','C1qb','P2ry12'],
    'CHOR': ['Ttr','Folr1','Kl','Sulf1'],
    'PER': ['Pltp','Ly6a','Bsg','Flt1'],
    'VEN': ['Vtn','Rgs5','Abcc9','Atp2a3'],
    'VLM': ['Ptgds', 'Myoc', 'Txnip', 'Gjb2'],
    'VSM': ['Myh11','Tagln','Mgp','Apod']

}

In [80]:
level_2_marker_list = []
for key in level_2_marker_dict.keys():
    level_2_marker_list= level_2_marker_list + level_2_marker_dict[key]
level_2_marker_list = list(dict.fromkeys(level_2_marker_list))

In [72]:
len(level_2_marker_list)

In [81]:
sc.pl.dotplot(
    adata,
    var_names=['Tbr1'],
    groupby='level_2',
    layer='log2_norm1e4_scaled',
    # values_to_plot='logfoldchanges',
    standard_scale="var",
    swap_axes=True,
    # dot_max=0.7,           # max dot size
    # cmap="YlGnBu_r",
    # save= 'level_3_marker.pdf'
)

In [88]:
sc.pl.dotplot(
    adata[adata.obs['level_2'] != 'Mix'],
    var_names=level_2_marker_list,
    groupby='level_2',
    layer='log2_norm1e4_scaled',
    # values_to_plot='logfoldchanges',
    standard_scale="var",
    swap_axes=True,
    dot_max=0.8,           # max dot size
    # cmap="YlGnBu_r",
    save= 'level_2_marker.pdf'
)

In [42]:
sc.tl.rank_genes_groups(adata, groupby='level_2',mask_var='highly_variable', method='wilcoxon', layer='log2_norm1e4_scaled')

In [10]:
sc.tl.filter_rank_genes_groups(adata, min_fold_change=.1, min_in_group_fraction=0.2, max_out_group_fraction=0.8)

In [12]:
sc.pl.rank_genes_groups_dotplot(adata, key='rank_genes_groups', n_genes=10, dendrogram=False)

In [43]:
sc.pl.rank_genes_groups_dotplot(adata, n_genes=10, dendrogram=False, standard_scale='var')

In [69]:
sc.pl.dotplot(
    adata,
    var_names=['Gad2'],
    groupby='level_2',
    layer='log2_norm1e4_scaled',
    # values_to_plot='logfoldchanges',
    standard_scale="var",
    swap_axes=True,
    # dot_max=0.7,           # max dot size
    # cmap="YlGnBu_r",
    # save= 'level_3_marker.pdf'
)

In [89]:
sc.pl.dotplot(
    adata[adata.obs['level_2'] != 'Mix'],
    var_names=level_3_marker_list,
    groupby='level_3',
    layer='log2_norm1e4_scaled',
    # values_to_plot='logfoldchanges',
    standard_scale="var",
    swap_axes=True,
    dot_max=0.8,           # max dot size
    # cmap="YlGnBu_r",
    save= 'level_3_marker.pdf'
)

# SI3

In [8]:
sc.tl.rank_genes_groups(adata, groupby='region_label', method='wilcoxon', layer='log2_norm1e4_scaled')

In [22]:
sc.pl.rank_genes_groups_dotplot(adata, key='rank_genes_groups', n_genes=15, dendrogram=False)

In [28]:
sc.pl.dotplot(
    adata,
    var_names=['C1ql3'],
    groupby='region_label',
    layer='log2_norm1e4_scaled',
    # values_to_plot='logfoldchanges',
    standard_scale="var",
    swap_axes=True,
    # dot_max=0.7,           # max dot size
    # cmap="YlGnBu_r",
    # save= 'level_3_marker.pdf'
)

In [23]:
rgn_marker_dict ={
    'CTX_L2/3': ['Lamp5'],
    'CTX_L4': ['Rorb'],
    'CTX_L5': ['Pcp4'],
    'CTX_L6': ['Hs3st4'],
    'CTX_ILA2/3': ['Hpcal4'],
    'CTX_mPFC5': ['Rab3c'],
    'CTXsp': ['Shank1'],
    'CTX_AI2/3': ['Cadm3'],
    'CTX_PIR': ['Chn1'],
    'CTX_CA1': ['Nr3c2'],
    'CTX_CA3': ['Neurod6'],
    'CTX_DG': ['Prox1'],
    'CTX_HIP': ['Aqp4'],

    'STR': ['Rasd2'],
    'PAL': ['Ly6h'],
    'TH_1': ['Prkcd'],
    'TH_2': ['Tcf7l2'],
    'TH_EPI': ['Gabbr1'],
    'TH_RT': ['Pvalb'],
    'HY': ['Nap1l5'],
    'FT': ['Qk'],
    'VS': ['Ttr'],
    'MNG': ['Ptgds'],
}


In [24]:
rgn_marker_list = []
for key in rgn_marker_dict.keys():
    rgn_marker_list= rgn_marker_list + rgn_marker_dict[key]
rgn_marker_list = list(dict.fromkeys(rgn_marker_list))

In [32]:
sc.pl.dotplot(
    adata,
    var_names=rgn_marker_list,
    groupby='region_label',
    layer='log2_norm1e4_scaled',
    # values_to_plot='logfoldchanges',
    standard_scale="var",
    swap_axes=True,
    # dot_max=0.8,           # max dot size
    # cmap="YlGnBu_r",
    save= 'rgn_marker.pdf'
)

#### cell type region composition

In [ ]:
adata.obs['coronal_position_label'] = adata.obs['coronal_position'].map({'PFC': 'CP1', 'ST': 'CP2', 'HP': 'CP3'})
adata.obs['coronal_position_label'] = adata.obs['coronal_position_label'].cat.reorder_categories(['CP3', 'CP2', 'CP1'])

In [ ]:
current_colors = [adata.uns['level_2_color_dict'][ctype] for ctype in adata.obs['level_2'].cat.categories[:-1]]
current_colors = list(reversed(current_colors))

In [ ]:
sns.set_style("ticks")

In [ ]:
# Create subplot with total counts on the right
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(5, 5), sharey=True, 
                               gridspec_kw={'width_ratios': [.9, .1]})

# Left subplot - proportion plot
sns.histplot(
    data=adata.obs.loc[adata.obs['level_2'] != 'Mix'],
    y="level_2", hue="coronal_position_label",
    multiple="fill", stat="proportion",
    discrete=True, shrink=.9, palette=cp_pl,
    legend=False,
    ax=ax1
)
ax1.set_xlabel('Proportion')
ax1.set_ylabel('level_2')

# row colors 
ax1.tick_params(axis='y', which='major', pad=15, length=0) 
for i, color in enumerate(current_colors):
    # ax1.set_xticks((-0.04, i*(1/len(current_colors))))
    ax1.add_patch(plt.Rectangle(xy=(-0.04, i*(1/len(current_colors))), width=.03, height=1/len(current_colors), color=color, lw=0,
                               transform=ax1.get_xaxis_transform(), clip_on=False))

# Right subplot - total counts
sns.histplot(
    data=adata.obs.loc[adata.obs['level_2'] != 'Mix'],
    y="level_2", 
    discrete=True, shrink=.9, 
    color='lightgray',
    ax=ax2,
)
ax2.vlines(x=50000, ymin=-0.4, ymax=15.4, color='gray', linewidth=1, linestyle='--')
ax2.tick_params(axis='y', which='major', pad=0, length=0) 
ax2.set_xscale('log')
ax2.set_xticks([])
ax2.set_xlabel('')

ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
ax2.spines['bottom'].set_visible(False)

plt.subplots_adjust(wspace=0)
plt.savefig(os.path.join(fig_path, 'l2_proportion.pdf'), dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
obs = adata.obs.loc[adata.obs['level_2'] != 'Mix', :].copy()
df_counts = pd.crosstab(obs['region_label'], obs['level_3_label'])

# Normalize across each row to get proportions
# df_counts_norm_region = df_counts.div(df_counts.sum(axis=1), axis=0)
df_counts_norm_ctype = df_counts.div(df_counts.sum(axis=0), axis=1)
# Calculate z-score for each column (cell type)
df_counts_norm_ctype_zscore = df_counts_norm_ctype.apply(lambda x: (x - x.mean()) / x.std(), axis=0)



In [ ]:
adata.uns['level_3_label_order'] = [
    'TEGLU_L2_3', 'TEGLU_L4_5', 'TEGLU_L5_6', 'TEGLU_L6', 'TEGLU_Mix', 'TEGLU_CA', 'DGGRC',
    'TEINH_1', 'TEINH_2', 'TEINH_3', 'TEINH_4',
    'MSN_1', 'MSN_2', 'PEP', 'DEGLU', 'HABCHO',
    'AC_1', 'AC_2', 'AC_3', 'OLG_1', 'OLG_2', 'OLG_3', 'OPC', 
    'CHOR', 'MGL', 'PER', 'VEN', 'VLM_1', 'VLM_2', 'VSM', 'Mix']

adata.obs['level_3_label'] = adata.obs['level_3_label'].cat.reorder_categories(adata.uns['level_3_label_order'])

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(df_counts_norm_ctype_zscore, cmap='bwr', cbar=False, 
            vmin=-4, vmax=4,
            ax=ax)
ax.axhline(y=9, color='black', linewidth=.5, alpha=1)
ax.axhline(y=13, color='black', linewidth=.5, alpha=1)
ax.axhline(y=15, color='black', linewidth=.5, alpha=1)
ax.axhline(y=19, color='black', linewidth=.5, alpha=1)


ax.axvline(x=6, color='black', linewidth=.5, alpha=1)
ax.axvline(x=7, color='black', linewidth=.5, alpha=1)
ax.axvline(x=11, color='black', linewidth=.5, alpha=1)
ax.axvline(x=13, color='black', linewidth=.5, alpha=1)
ax.axvline(x=16, color='black', linewidth=.5, alpha=1)
ax.axvline(x=19, color='black', linewidth=.5, alpha=1)
ax.axvline(x=22, color='black', linewidth=.5, alpha=1)
ax.axvline(x=27, color='black', linewidth=.5, alpha=1)
ax.axvline(x=29, color='black', linewidth=.5, alpha=1)

ax.spines['right'].set_visible(True)
ax.spines['left'].set_visible(True)
ax.spines['top'].set_visible(True)
ax.spines['bottom'].set_visible(True)

plt.savefig(os.path.join(fig_path, 'l3_region_composition_h.pdf'), dpi=100, bbox_inches='tight')
